In [ ]:
# 1. First, install and import the necessary libraries
!pip install --quiet ONE-api
!pip install --quiet ibllib

import os
os.environ.setdefault('ONE_HTTP_DL_THREADS', '1')

from one.api import ONE
import warnings
warnings.simplefilter("ignore", FutureWarning)

# 2. Initialize the ONE API
ONE.setup(base_url='https://openalyx.internationalbrainlab.org', silent=True)
one = ONE(password='international')

# 3. Put your Session IDs into a list
session_ids = [
    '5569f363-0934-464e-9a5b-77c8e67791a1',
    '1191f865-b10a-45c8-9c48-24a980fd9402',
    '360eac0c-7d2d-4cc1-9dcf-79fc7afc56e7',
    '8207abc6-6b23-4762-92b4-82e05bed5143',
    '0f77ca5d-73c2-45bd-aa4c-4c5ed275dbde',
    '5455a21c-1be7-4cae-ae8e-8853a8d5f55e',
    '51e53aff-1d5d-4182-a684-aba783d50ae5',
    'd23a44ef-1402-4ed7-97f5-47e9a7a504d9',
    '6ed57216-498d-48a6-b48b-a243a34710ea',
    'ecb5520d-1358-434c-95ec-93687ecd1396',
    'ca4ecb4c-4b60-4723-9b9e-2c54a6290a53',
    '3f859b5c-e73a-4044-b49e-34bb81e96715',
    'db4df448-e449-4a6f-a0e7-288711e7a75a',
    'e349a2e7-50a3-47ca-bc45-20d1899854ec',
    '5339812f-8b91-40ba-9d8f-a559563cc46b',
    '1ec23a70-b94b-4e9c-a0df-8c2151da3761',
    'f88d4dd4-ccd7-400e-9035-fa00be3bcfa8',
    '2038e95d-64d4-4ecb-83d0-1308d3c598f8',
    '0cc486c3-8c7b-494d-aa04-b70e2690bcba',
    '0f25376f-2b78-4ddc-8c39-b6cdbe7bf5b9'
]

# 4. To load data from a session (example):
eid = session_ids[0]  # Let's select the first session

# To load spike data:
spikes = one.load_object(eid, 'spikes')

# Or to load a specific dataset:
spike_times = one.load_dataset(eid, 'spikes.times.npy')
spike_clusters = one.load_dataset(eid, 'spikes.clusters.npy')

# 5. If you want to load data for all sessions using a loop:
for eid in session_ids:
    try:
        spikes = one.load_object(eid, 'spikes')
        print(f"Session {eid} loaded")
        # You can perform desired operations with the data here
    except Exception as e:
        print(f"Error while loading session {eid}: {e}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
import pandas as pd

# 1. iSTTC Function
def calculate_isttc(spike_times, delta_t=0.005, max_lag=1.0):
    """
    Calculates the Inter-Spike-Time Tiling Coefficient
    
    Parameters:
    -----------
    spike_times : array
        Spike times (in seconds)
    delta_t : float
        Time window (default 5ms = 0.005s)
    max_lag : float
        Maximum lag time (seconds)
    
    Returns:
    --------
    lags : array
        Lag times
    isttc : array
        iSTTC values for each lag
    """
    
    # Calculate ISI (Inter-Spike Interval)
    isis = np.diff(spike_times)
    
    # Create lag values
    lags = np.arange(0, max_lag, delta_t)
    isttc = np.zeros(len(lags))
    
    for i, lag in enumerate(lags):
        # For each ISI, check if there is another spike at the lag time
        if len(isis) > 0:
            # Tiling: Count spikes falling within the lag+delta_t range inside ISIs
            tiles = np.sum((isis >= lag) & (isis < lag + delta_t))
            isttc[i] = tiles / len(isis)
    
    return lags, isttc


# 2. Exponential Fit Function
def exponential_decay(t, a, tau, c):
    """
    Exponential decay: a * exp(-t/tau) + c
    tau = intrinsic timescale
    """
    return a * np.exp(-t / tau) + c


# 3. Intrinsic Timescale Calculation
def calculate_intrinsic_timescale(spike_times, delta_t=0.005, max_lag=1.0, 
                                   fit_range=(0.01, 0.5)):
    """
    Calculates intrinsic timescale using iSTTC
    
    Returns:
    --------
    tau : float
        Intrinsic timescale (seconds)
    lags : array
        Lag times
    isttc : array
        iSTTC values
    fit_curve : array
        Fitted exponential curve
    """
    
    # Calculate iSTTC
    lags, isttc = calculate_isttc(spike_times, delta_t, max_lag)
    
    # Select the range to be used for fitting
    fit_mask = (lags >= fit_range[0]) & (lags <= fit_range[1])
    lags_fit = lags[fit_mask]
    isttc_fit = isttc[fit_mask]
    
    # Perform exponential fit
    try:
        # Initial guess
        p0 = [np.max(isttc_fit), 0.1, np.min(isttc_fit)]
        popt, _ = curve_fit(exponential_decay, lags_fit, isttc_fit, p0=p0, maxfev=10000)
        tau = popt[1]  # Intrinsic timescale
        
        # Calculate fit curve
        fit_curve = exponential_decay(lags, *popt)
        
    except:
        tau = np.nan
        fit_curve = np.full_like(lags, np.nan)
    
    return tau, lags, isttc, fit_curve


# 4. Visualization Function
def plot_example_neuron(spike_times, neuron_id="Example"):
    """
    Visualize iSTTC and fit for a single neuron
    """
    tau, lags, isttc, fit_curve = calculate_intrinsic_timescale(spike_times)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Left: Raster plot
    axes[0].eventplot(spike_times[:1000], linewidths=0.5)
    axes[0].set_xlabel('Time (s)')
    axes[0].set_ylabel('Spike')
    axes[0].set_title(f'{neuron_id}: Spike Raster (first 1000 spikes)')
    
    # Right: iSTTC and fit
    axes[1].plot(lags * 1000, isttc, 'o-', markersize=3, label='iSTTC')
    axes[1].plot(lags * 1000, fit_curve, 'r-', linewidth=2, 
                label=f'Fit (τ = {tau*1000:.1f} ms)')
    axes[1].set_xlabel('Lag (ms)')
    axes[1].set_ylabel('iSTTC')
    axes[1].set_title(f'{neuron_id}: Intrinsic Timescale')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    return tau

print("Functions defined! You can now run the test code.")

In [ ]:
# 1. Let's use the first successfully loaded session
test_eid = '5569f363-0934-464e-9a5b-77c8e67791a1'  # First successful session

# 2. Load spike data
print(f"Loading session {test_eid}...")
spikes = one.load_object(test_eid, 'spikes', collection='alf/probe00/pykilosort')

print(f"\nLoaded data keys: {spikes.keys()}")
print(f"Total number of spikes: {len(spikes['times'])}")

# 3. Find unique clusters
unique_clusters = np.unique(spikes['clusters'])
print(f"Total number of clusters: {len(unique_clusters)}")

# 4. Select a good neuron based on firing rate (one that doesn't fire too little or too much)
cluster_stats = []
for cluster_id in unique_clusters[:50]:  # Check the first 50 clusters
    cluster_mask = spikes['clusters'] == cluster_id
    spike_times = spikes['times'][cluster_mask]
    
    if len(spike_times) > 100:
        duration = spike_times[-1] - spike_times[0]
        firing_rate = len(spike_times) / duration
        cluster_stats.append({
            'cluster_id': cluster_id,
            'n_spikes': len(spike_times),
            'firing_rate': firing_rate
        })

# Select a neuron with a firing rate between 2-20 Hz (a reasonable range)
cluster_stats_df = pd.DataFrame(cluster_stats)
good_neurons = cluster_stats_df[(cluster_stats_df['firing_rate'] >= 2) & 
                                 (cluster_stats_df['firing_rate'] <= 20)]

if len(good_neurons) > 0:
    selected_cluster = good_neurons.iloc[0]['cluster_id']
    print(f"\n=== Selected Neuron ===")
    print(f"Cluster ID: {selected_cluster}")
    print(f"Number of spikes: {good_neurons.iloc[0]['n_spikes']:.0f}")
    print(f"Firing rate: {good_neurons.iloc[0]['firing_rate']:.2f} Hz")
else:
    selected_cluster = unique_clusters[0]
    print(f"\n=== Default Neuron (first cluster) ===")
    print(f"Cluster ID: {selected_cluster}")

# 5. Get the spikes for this neuron
cluster_mask = spikes['clusters'] == selected_cluster
spike_times = spikes['times'][cluster_mask]

print(f"\nAnalysis starting...")

# 6. Calculate iSTTC and visualize
tau = plot_example_neuron(spike_times, neuron_id=f"Cluster {selected_cluster}")

print(f"\n=== RESULT ===")
print(f"Intrinsic Timescale (τ): {tau*1000:.2f} ms")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
import pandas as pd

# OPTION 1: Analyze all neurons in a session
def analyze_all_neurons_in_session(eid, one, probe='probe00'):
    """
    Calculate the timescales of all neurons in a session
    """
    spikes = one.load_object(eid, 'spikes', collection=f'alf/{probe}/pykilosort')
    unique_clusters = np.unique(spikes['clusters'])
    
    results = []
    
    print(f"Total number of clusters: {len(unique_clusters)}")
    
    for i, cluster_id in enumerate(unique_clusters):
        if i % 50 == 0:
            print(f"Processing: {i}/{len(unique_clusters)}")
        
        cluster_mask = spikes['clusters'] == cluster_id
        spike_times = spikes['times'][cluster_mask]
        
        if len(spike_times) < 100:
            continue
        
        duration = spike_times[-1] - spike_times[0]
        firing_rate = len(spike_times) / duration
        
        tau, _, _, _ = calculate_intrinsic_timescale(spike_times)
        
        if not np.isnan(tau) and 0.01 < tau < 2:  # Between 10ms - 2s
            results.append({
                'cluster_id': cluster_id,
                'tau_ms': tau * 1000,
                'firing_rate': firing_rate,
                'n_spikes': len(spike_times)
            })
    
    return pd.DataFrame(results)


# OPTION 2: Visualize distribution with histograms
def plot_timescale_distribution(results_df):
    """
    Visualize the timescale distribution
    """
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Timescale histogram
    axes[0, 0].hist(results_df['tau_ms'], bins=50, edgecolor='black', alpha=0.7)
    axes[0, 0].set_xlabel('Intrinsic Timescale (ms)')
    axes[0, 0].set_ylabel('Number of Neurons')
    axes[0, 0].set_title('Timescale Distribution')
    axes[0, 0].axvline(results_df['tau_ms'].median(), color='r', 
                        linestyle='--', label=f'Median: {results_df["tau_ms"].median():.1f} ms')
    axes[0, 0].legend()
    
    # Firing rate histogram
    axes[0, 1].hist(results_df['firing_rate'], bins=50, edgecolor='black', alpha=0.7)
    axes[0, 1].set_xlabel('Firing Rate (Hz)')
    axes[0, 1].set_ylabel('Number of Neurons')
    axes[0, 1].set_title('Firing Rate Distribution')
    
    # Timescale vs Firing Rate
    axes[1, 0].scatter(results_df['firing_rate'], results_df['tau_ms'], alpha=0.5)
    axes[1, 0].set_xlabel('Firing Rate (Hz)')
    axes[1, 0].set_ylabel('Intrinsic Timescale (ms)')
    axes[1, 0].set_title('Timescale vs Firing Rate')
    axes[1, 0].set_xscale('log')
    
    # Log-scale timescale histogram
    axes[1, 1].hist(np.log10(results_df['tau_ms']), bins=50, edgecolor='black', alpha=0.7)
    axes[1, 1].set_xlabel('Log10(Timescale) (ms)')
    axes[1, 1].set_ylabel('Number of Neurons')
    axes[1, 1].set_title('Timescale Distribution (Log Scale)')
    
    plt.tight_layout()
    plt.show()


# OPTION 3: Compare neurons with different firing rates
def compare_neurons_by_firing_rate(results_df):
    """
    Compare the timescales of different firing rate groups
    """
    # Group by firing rate
    results_df['fr_group'] = pd.cut(results_df['firing_rate'], 
                                     bins=[0, 2, 10, 50], 
                                     labels=['Low (0-2Hz)', 'Medium (2-10Hz)', 'High (10-50Hz)'])
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    for group in results_df['fr_group'].unique():
        if pd.notna(group):
            data = results_df[results_df['fr_group'] == group]['tau_ms']
            ax.hist(data, bins=30, alpha=0.5, label=group)
    
    ax.set_xlabel('Intrinsic Timescale (ms)')
    ax.set_ylabel('Number of Neurons')
    ax.set_title('Timescale Distribution: By Firing Rate Groups')
    ax.legend()
    plt.show()


print("✓ All functions defined!")

In [ ]:
# 1. Analyze all neurons
print("=== Analyzing All Neurons in the Session ===\n")
test_eid = '5569f363-0934-464e-9a5b-77c8e67791a1'

results_df = analyze_all_neurons_in_session(test_eid, one, probe='probe00')

print(f"\n=== Analysis Complete ===")
print(f"Total number of neurons analyzed: {len(results_df)}")
print(f"\n--- Statistics ---")
print(f"Average timescale: {results_df['tau_ms'].mean():.2f} ms")
print(f"Median timescale: {results_df['tau_ms'].median():.2f} ms")
print(f"Min timescale: {results_df['tau_ms'].min():.2f} ms")
print(f"Max timescale: {results_df['tau_ms'].max():.2f} ms")
print(f"\nAverage firing rate: {results_df['firing_rate'].mean():.2f} Hz")
print(f"Median firing rate: {results_df['firing_rate'].median():.2f} Hz")

# 2. Visualize the distribution
print("\nCreating visualization...")
plot_timescale_distribution(results_df)

# 3. Show the first 10 neurons
print("\n=== First 10 Neurons ===")
print(results_df.head(10).to_string(index=False))

# 4. Neurons with the shortest and longest timescales
print("\n=== 5 Neurons with the Shortest Timescale ===")
print(results_df.nsmallest(5, 'tau_ms')[['cluster_id', 'tau_ms', 'firing_rate']].to_string(index=False))

print("\n=== 5 Neurons with the Longest Timescale ===")
print(results_df.nlargest(5, 'tau_ms')[['cluster_id', 'tau_ms', 'firing_rate']].to_string(index=False))

# 5. Compare by firing rate groups
print("\nComparing by firing rate groups...")
compare_neurons_by_firing_rate(results_df)

In [ ]:

# 1. Multi-Timescale Exponential Decay (Shi et al. approach)
def multi_exponential_decay(t, a1, tau1, a2, tau2, c):
    """
    Two-component exponential decay:
    a1 * exp(-t/tau1) + a2 * exp(-t/tau2) + c
    
    tau1: fast timescale (short-term dynamics)
    tau2: slow timescale (long-term dynamics)
    """
    return a1 * np.exp(-t / tau1) + a2 * np.exp(-t / tau2) + c


def single_exponential_decay(t, a, tau, c):
    """
    Single-component exponential decay (for comparison)
    """
    return a * np.exp(-t / tau) + c


# 2. Enhanced iSTTC Calculation - With Multi-Timescale Fit
def calculate_intrinsic_timescales_multi(spike_times, delta_t=0.005, max_lag=1.0, 
                                          fit_range=(0.01, 0.5)):
    """
    Calculates MULTIPLE intrinsic timescales using iSTTC
    
    Returns:
    --------
    results : dict
        - tau_fast: Fast timescale (seconds)
        - tau_slow: Slow timescale (seconds)
        - tau_single: Timescale from single exponential fit
        - a1, a2: Weight of each component
        - r2_single: R² for single fit
        - r2_multi: R² for multi fit
        - lags, isttc, fit_single, fit_multi
    """
    
    # Calculate ISI
    isis = np.diff(spike_times)
    
    # Create lag values
    lags = np.arange(0, max_lag, delta_t)
    isttc = np.zeros(len(lags))
    
    for i, lag in enumerate(lags):
        if len(isis) > 0:
            tiles = np.sum((isis >= lag) & (isis < lag + delta_t))
            isttc[i] = tiles / len(isis)
    
    # Select range for fitting
    fit_mask = (lags >= fit_range[0]) & (lags <= fit_range[1])
    lags_fit = lags[fit_mask]
    isttc_fit = isttc[fit_mask]
    
    results = {
        'tau_fast': np.nan,
        'tau_slow': np.nan,
        'tau_single': np.nan,
        'a1': np.nan,
        'a2': np.nan,
        'r2_single': np.nan,
        'r2_multi': np.nan,
        'lags': lags,
        'isttc': isttc,
        'fit_single': np.full_like(lags, np.nan),
        'fit_multi': np.full_like(lags, np.nan)
    }
    
    # SINGLE EXPONENTIAL FIT
    try:
        p0_single = [np.max(isttc_fit), 0.1, np.min(isttc_fit)]
        popt_single, _ = curve_fit(single_exponential_decay, lags_fit, isttc_fit, 
                                    p0=p0_single, maxfev=10000)
        results['tau_single'] = popt_single[1]
        results['fit_single'] = single_exponential_decay(lags, *popt_single)
        
        # Calculate R²
        ss_res = np.sum((isttc_fit - single_exponential_decay(lags_fit, *popt_single))**2)
        ss_tot = np.sum((isttc_fit - np.mean(isttc_fit))**2)
        results['r2_single'] = 1 - (ss_res / ss_tot)
    except:
        pass
    
    # MULTI EXPONENTIAL FIT (2 timescales)
    try:
        # Initial guesses: fast (~50ms), slow (~200ms)
        p0_multi = [
            np.max(isttc_fit) * 0.6,  # a1 (fast component)
            0.05,                       # tau1 (fast: ~50ms)
            np.max(isttc_fit) * 0.4,  # a2 (slow component)
            0.2,                        # tau2 (slow: ~200ms)
            np.min(isttc_fit)           # c (baseline)
        ]
        
        # Bounds to ensure tau1 < tau2
        bounds = (
            [0, 0.01, 0, 0.05, 0],      # Lower bounds
            [np.inf, 0.15, np.inf, 2, np.inf]  # Upper bounds
        )
        
        popt_multi, _ = curve_fit(multi_exponential_decay, lags_fit, isttc_fit, 
                                   p0=p0_multi, bounds=bounds, maxfev=10000)
        
        results['a1'] = popt_multi[0]
        results['tau_fast'] = popt_multi[1]
        results['a2'] = popt_multi[2]
        results['tau_slow'] = popt_multi[3]
        results['fit_multi'] = multi_exponential_decay(lags, *popt_multi)
        
        # Calculate R²
        ss_res = np.sum((isttc_fit - multi_exponential_decay(lags_fit, *popt_multi))**2)
        ss_tot = np.sum((isttc_fit - np.mean(isttc_fit))**2)
        results['r2_multi'] = 1 - (ss_res / ss_tot)
    except:
        pass
    
    return results


# 3. Enhanced Visualization
def plot_example_neuron_multi(spike_times, neuron_id="Example"):
    """
    Visualize multi-timescale analysis
    """
    results = calculate_intrinsic_timescales_multi(spike_times)
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # Left: Raster plot
    axes[0].eventplot(spike_times[:1000], linewidths=0.5)
    axes[0].set_xlabel('Time (s)')
    axes[0].set_ylabel('Spike')
    axes[0].set_title(f'{neuron_id}: Spike Raster (first 1000 spikes)')
    
    # Middle: Single vs Multi Fit Comparison
    axes[1].plot(results['lags'] * 1000, results['isttc'], 'o', 
                 markersize=3, label='iSTTC data', alpha=0.5)
    
    if not np.isnan(results['tau_single']):
        axes[1].plot(results['lags'] * 1000, results['fit_single'], 'b-', 
                    linewidth=2, label=f'Single (τ={results["tau_single"]*1000:.1f}ms, R²={results["r2_single"]:.3f})')
    
    if not np.isnan(results['tau_fast']):
        axes[1].plot(results['lags'] * 1000, results['fit_multi'], 'r-', 
                    linewidth=2, 
                    label=f'Multi (τ_fast={results["tau_fast"]*1000:.1f}ms, τ_slow={results["tau_slow"]*1000:.1f}ms, R²={results["r2_multi"]:.3f})')
    
    axes[1].set_xlabel('Lag (ms)')
    axes[1].set_ylabel('iSTTC')
    axes[1].set_title(f'{neuron_id}: Single vs Multi Timescale')
    axes[1].legend(fontsize=8)
    axes[1].grid(True, alpha=0.3)
    
    # Right: Residuals (fit quality)
    if not np.isnan(results['tau_single']) and not np.isnan(results['tau_fast']):
        residual_single = results['isttc'] - results['fit_single']
        residual_multi = results['isttc'] - results['fit_multi']
        
        axes[2].plot(results['lags'] * 1000, residual_single, 'b-', 
                    linewidth=1, label='Single residuals', alpha=0.7)
        axes[2].plot(results['lags'] * 1000, residual_multi, 'r-', 
                    linewidth=1, label='Multi residuals', alpha=0.7)
        axes[2].axhline(0, color='k', linestyle='--', alpha=0.3)
        axes[2].set_xlabel('Lag (ms)')
        axes[2].set_ylabel('Residuals')
        axes[2].set_title('Fit Quality Comparison')
        axes[2].legend()
        axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    return results


# 4. Analyze all neurons with Multi-Timescale
def analyze_all_neurons_multi(eid, one, probe='probe00'):
    """
    Analyze all neurons in a session using the multi-timescale approach
    """
    spikes = one.load_object(eid, 'spikes', collection=f'alf/{probe}/pykilosort')
    unique_clusters = np.unique(spikes['clusters'])
    
    results_list = []
    
    print(f"Total number of clusters: {len(unique_clusters)}")
    
    for i, cluster_id in enumerate(unique_clusters):
        if i % 50 == 0:
            print(f"Processing: {i}/{len(unique_clusters)}")
        
        cluster_mask = spikes['clusters'] == cluster_id
        spike_times = spikes['times'][cluster_mask]
        
        if len(spike_times) < 100:
            continue
        
        duration = spike_times[-1] - spike_times[0]
        firing_rate = len(spike_times) / duration
        
        # Multi-timescale analysis
        res = calculate_intrinsic_timescales_multi(spike_times)
        
        # Save only valid results
        if not np.isnan(res['tau_single']):
            results_list.append({
                'cluster_id': cluster_id,
                'firing_rate': firing_rate,
                'n_spikes': len(spike_times),
                'tau_single_ms': res['tau_single'] * 1000,
                'tau_fast_ms': res['tau_fast'] * 1000 if not np.isnan(res['tau_fast']) else np.nan,
                'tau_slow_ms': res['tau_slow'] * 1000 if not np.isnan(res['tau_slow']) else np.nan,
                'a1': res['a1'],
                'a2': res['a2'],
                'r2_single': res['r2_single'],
                'r2_multi': res['r2_multi'],
                'delta_r2': res['r2_multi'] - res['r2_single'] if not np.isnan(res['r2_multi']) else np.nan
            })
    
    return pd.DataFrame(results_list)


# 5. Enhanced Visualization
def plot_multi_timescale_results(results_df):
    """
    Visualize multi-timescale analysis results
    """
    fig = plt.figure(figsize=(18, 12))
    
    # 1. Single vs Multi Fit Comparison (R²)
    ax1 = plt.subplot(3, 3, 1)
    valid_multi = results_df.dropna(subset=['r2_multi'])
    ax1.scatter(valid_multi['r2_single'], valid_multi['r2_multi'], alpha=0.5)
    ax1.plot([0, 1], [0, 1], 'r--', label='Identity')
    ax1.set_xlabel('R² Single')
    ax1.set_ylabel('R² Multi')
    ax1.set_title('Fit Quality: Single vs Multi')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 2. Delta R² distribution
    ax2 = plt.subplot(3, 3, 2)
    delta_r2_valid = results_df['delta_r2'].dropna()
    ax2.hist(delta_r2_valid, bins=50, edgecolor='black', alpha=0.7)
    ax2.axvline(0, color='r', linestyle='--')
    ax2.set_xlabel('ΔR² (Multi - Single)')
    ax2.set_ylabel('Count')
    ax2.set_title(f'Multi Fit Improvement (Mean ΔR²={delta_r2_valid.mean():.4f})')
    
    # 3. Neurons needing multi-timescale
    ax3 = plt.subplot(3, 3, 3)
    improvement_threshold = 0.05
    n_improved = np.sum(results_df['delta_r2'] > improvement_threshold)
    n_total = len(results_df.dropna(subset=['delta_r2']))
    labels = [f'Multi better\n(ΔR²>0.05)\n{n_improved}', f'Single sufficient\n{n_total - n_improved}']
    sizes = [n_improved, n_total - n_improved]
    ax3.pie(sizes, labels=labels, autopct='%1.1f%%', startangle=90)
    ax3.set_title('Multi-Timescale Necessity')
    
    # 4. Fast vs Slow Timescale
    ax4 = plt.subplot(3, 3, 4)
    valid_both = results_df.dropna(subset=['tau_fast_ms', 'tau_slow_ms'])
    ax4.scatter(valid_both['tau_fast_ms'], valid_both['tau_slow_ms'], alpha=0.5)
    ax4.set_xlabel('Fast Timescale (ms)')
    ax4.set_ylabel('Slow Timescale (ms)')
    ax4.set_title('Fast vs Slow Timescales')
    ax4.set_xscale('log')
    ax4.set_yscale('log')
    ax4.grid(True, alpha=0.3)
    
    # 5. Timescale distributions
    ax5 = plt.subplot(3, 3, 5)
    ax5.hist(results_df['tau_single_ms'], bins=50, alpha=0.5, label='Single', edgecolor='black')
    ax5.hist(valid_both['tau_fast_ms'], bins=50, alpha=0.5, label='Fast', edgecolor='black')
    ax5.hist(valid_both['tau_slow_ms'], bins=50, alpha=0.5, label='Slow', edgecolor='black')
    ax5.set_xlabel('Timescale (ms)')
    ax5.set_ylabel('Count')
    ax5.set_title('Timescale Distributions')
    ax5.legend()
    ax5.set_xlim(0, 500)
    
    # 6. Component weights (a1 vs a2)
    ax6 = plt.subplot(3, 3, 6)
    valid_weights = results_df.dropna(subset=['a1', 'a2'])
    ax6.scatter(valid_weights['a1'], valid_weights['a2'], alpha=0.5)
    ax6.set_xlabel('Fast Component Weight (a1)')
    ax6.set_ylabel('Slow Component Weight (a2)')
    ax6.set_title('Component Weights')
    ax6.grid(True, alpha=0.3)
    
    # 7. Firing Rate vs Fast Timescale
    ax7 = plt.subplot(3, 3, 7)
    ax7.scatter(valid_both['firing_rate'], valid_both['tau_fast_ms'], alpha=0.5)
    ax7.set_xlabel('Firing Rate (Hz)')
    ax7.set_ylabel('Fast Timescale (ms)')
    ax7.set_title('Firing Rate vs Fast Timescale')
    ax7.set_xscale('log')
    ax7.grid(True, alpha=0.3)
    
    # 8. Firing Rate vs Slow Timescale
    ax8 = plt.subplot(3, 3, 8)
    ax8.scatter(valid_both['firing_rate'], valid_both['tau_slow_ms'], alpha=0.5)
    ax8.set_xlabel('Firing Rate (Hz)')
    ax8.set_ylabel('Slow Timescale (ms)')
    ax8.set_title('Firing Rate vs Slow Timescale')
    ax8.set_xscale('log')
    ax8.grid(True, alpha=0.3)
    
    # 9. Timescale ratio
    ax9 = plt.subplot(3, 3, 9)
    valid_both['ratio'] = valid_both['tau_slow_ms'] / valid_both['tau_fast_ms']
    ax9.hist(valid_both['ratio'], bins=50, edgecolor='black', alpha=0.7)
    ax9.set_xlabel('Slow/Fast Ratio')
    ax9.set_ylabel('Count')
    ax9.set_title(f'Timescale Separation (Median={valid_both["ratio"].median():.1f}x)')
    
    plt.tight_layout()
    plt.show()


print("✓ Multi-timescale functions defined!")
print("\nYou can now run the following:")
print("1. Single neuron test: plot_example_neuron_multi(spike_times, 'Cluster 0')")
print("2. Entire session: results_df = analyze_all_neurons_multi(test_eid, one)")
print("3. Visualization: plot_multi_timescale_results(results_df)")

In [ ]:
# STEP 1: First define the multi-timescale functions
# (Run the long code provided earlier - until "✓ Multi-timescale functions defined!" appears)

# STEP 2: Analyze the entire session
print("=== Starting Multi-Timescale Analysis ===\n")
test_eid = '5569f363-0934-464e-9a5b-77c8e67791a1'

results_df = analyze_all_neurons_multi(test_eid, one, probe='probe00')

print(f"\n=== Analysis Complete ===")
print(f"Total number of neurons analyzed: {len(results_df)}")

# STEP 3: View basic statistics
print(f"\n--- Single Timescale Statistics ---")
print(f"Average: {results_df['tau_single_ms'].mean():.2f} ms")
print(f"Median: {results_df['tau_single_ms'].median():.2f} ms")

# Neurons with valid multi-timescale fits
multi_valid = results_df.dropna(subset=['tau_fast_ms', 'tau_slow_ms'])
print(f"\n--- Multi-Timescale Statistics ---")
print(f"Number of neurons with multi-timescale fits: {len(multi_valid)}/{len(results_df)}")
if len(multi_valid) > 0:
    print(f"Fast timescale - Average: {multi_valid['tau_fast_ms'].mean():.2f} ms")
    print(f"Slow timescale - Average: {multi_valid['tau_slow_ms'].mean():.2f} ms")
    print(f"Average R² improvement: {multi_valid['delta_r2'].mean():.4f}")

# STEP 4: Create visualizations
print("\n=== Creating Visualizations ===")
plot_multi_timescale_results(results_df)

# STEP 6: Examine example neurons
print("\n=== Best Multi-Timescale Examples (Highest R² Improvement) ===")
best_multi = results_df.nlargest(5, 'delta_r2')[['cluster_id', 'tau_fast_ms', 'tau_slow_ms', 'delta_r2', 'r2_multi']]
print(best_multi.to_string(index=False))

In [ ]:
# Successfully loaded sessions (those that did not return an error)
successful_sessions = [
    '5569f363-0934-464e-9a5b-77c8e67791a1',
    '5455a21c-1be7-4cae-ae8e-8853a8d5f55e',
    'ca4ecb4c-4b60-4723-9b9e-2c54a6290a53',
    'db4df448-e449-4a6f-a0e7-288711e7a75a',
    '2038e95d-64d4-4ecb-83d0-1308d3c598f8',
    '0f25376f-2b78-4ddc-8c39-b6cdbe7bf5b9'
]

print(f"=== {len(successful_sessions)} Sessions to be Analyzed ===\n")

# To combine all results
all_results = []

for i, eid in enumerate(successful_sessions, 1):
    print(f"\n{'='*60}")
    print(f"Session {i}/{len(successful_sessions)}: {eid}")
    print(f"{'='*60}")
    
    try:
        # Analyze the session
        session_results = analyze_all_neurons_multi(eid, one, probe='probe00')
        
        # Add Session ID
        session_results['session_id'] = eid
        session_results['session_num'] = i
        
        # Add to the list
        all_results.append(session_results)
        
        print(f"\n✓ Success! {len(session_results)} neurons analyzed")
        print(f"  - Multi-timescale fitted: {session_results['tau_fast_ms'].notna().sum()}")
        print(f"  - Average tau_single: {session_results['tau_single_ms'].mean():.2f} ms")
        
        if session_results['tau_fast_ms'].notna().sum() > 0:
            valid_multi = session_results.dropna(subset=['tau_fast_ms'])
            print(f"  - Average tau_fast: {valid_multi['tau_fast_ms'].mean():.2f} ms")
            print(f"  - Average tau_slow: {valid_multi['tau_slow_ms'].mean():.2f} ms")
            print(f"  - Average ΔR²: {valid_multi['delta_r2'].mean():.4f}")
        
    except Exception as e:
        print(f"\n✗ Error: {str(e)}")
        continue

# Combine all results
print(f"\n\n{'='*60}")
print("=== COMBINING ALL SESSIONS ===")
print(f"{'='*60}")

combined_results = pd.concat(all_results, ignore_index=True)

print(f"\n=== GENERAL STATISTICS ===")
print(f"Total number of sessions: {len(successful_sessions)}")
print(f"Total number of neurons: {len(combined_results)}")
print(f"Average neurons per session: {len(combined_results)/len(successful_sessions):.1f}")

print(f"\n--- Single Timescale ---")
print(f"Average: {combined_results['tau_single_ms'].mean():.2f} ms")
print(f"Median: {combined_results['tau_single_ms'].median():.2f} ms")
print(f"Std Dev: {combined_results['tau_single_ms'].std():.2f} ms")
print(f"Min: {combined_results['tau_single_ms'].min():.2f} ms")
print(f"Max: {combined_results['tau_single_ms'].max():.2f} ms")

print(f"\n--- Multi Timescale ---")
multi_valid = combined_results.dropna(subset=['tau_fast_ms', 'tau_slow_ms'])
print(f"Multi-timescale fitted: {len(multi_valid)}/{len(combined_results)} ({len(multi_valid)/len(combined_results)*100:.1f}%)")

if len(multi_valid) > 0:
    print(f"\nFast Timescale:")
    print(f"  Average: {multi_valid['tau_fast_ms'].mean():.2f} ms")
    print(f"  Median: {multi_valid['tau_fast_ms'].median():.2f} ms")
    print(f"  Std Dev: {multi_valid['tau_fast_ms'].std():.2f} ms")
    
    print(f"\nSlow Timescale:")
    print(f"  Average: {multi_valid['tau_slow_ms'].mean():.2f} ms")
    print(f"  Median: {multi_valid['tau_slow_ms'].median():.2f} ms")
    print(f"  Std Dev: {multi_valid['tau_slow_ms'].std():.2f} ms")
    
    print(f"\nFit Quality:")
    print(f"  Average R² improvement: {multi_valid['delta_r2'].mean():.4f}")
    print(f"  Number of neurons with ΔR² > 0.05: {(multi_valid['delta_r2'] > 0.05).sum()} ({(multi_valid['delta_r2'] > 0.05).sum()/len(combined_results)*100:.1f}%)")

print(f"\n--- Firing Rate ---")
print(f"Average: {combined_results['firing_rate'].mean():.2f} Hz")
print(f"Median: {combined_results['firing_rate'].median():.2f} Hz")
print(f"Std Dev: {combined_results['firing_rate'].std():.2f} Hz")
print(f"Min: {combined_results['firing_rate'].min():.2f} Hz")
print(f"Max: {combined_results['firing_rate'].max():.2f} Hz")


# Visualize
print(f"\n{'='*60}")
print("=== CREATING VISUALIZATIONS ===")
print(f"{'='*60}")

plot_multi_timescale_results(combined_results)

# Comparison between sessions
print("\n=== INTER-SESSION COMPARISON ===")
session_summary = combined_results.groupby('session_num').agg({
    'tau_single_ms': ['mean', 'median', 'std'],
    'tau_fast_ms': ['mean', 'count'],
    'tau_slow_ms': 'mean',
    'delta_r2': 'mean',
    'firing_rate': 'mean'
}).round(2)

print(session_summary)

# Inter-session comparison plot
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1. Distribution of tau_single by session
ax1 = axes[0, 0]
combined_results.boxplot(column='tau_single_ms', by='session_num', ax=ax1)
ax1.set_xlabel('Session')
ax1.set_ylabel('Single Timescale (ms)')
ax1.set_title('Timescale Across Sessions')
plt.sca(ax1)
plt.xticks(rotation=0)

# 2. Fast timescale by session
ax2 = axes[0, 1]
multi_valid.boxplot(column='tau_fast_ms', by='session_num', ax=ax2)
ax2.set_xlabel('Session')
ax2.set_ylabel('Fast Timescale (ms)')
ax2.set_title('Fast Timescale Across Sessions')
plt.sca(ax2)
plt.xticks(rotation=0)

# 3. Slow timescale by session
ax3 = axes[0, 2]
multi_valid.boxplot(column='tau_slow_ms', by='session_num', ax=ax3)
ax3.set_xlabel('Session')
ax3.set_ylabel('Slow Timescale (ms)')
ax3.set_title('Slow Timescale Across Sessions')
plt.sca(ax3)
plt.xticks(rotation=0)

# 4. Number of neurons by session
ax4 = axes[1, 0]
neuron_counts = combined_results.groupby('session_num').size()
ax4.bar(neuron_counts.index, neuron_counts.values)
ax4.set_xlabel('Session')
ax4.set_ylabel('Number of Neurons')
ax4.set_title('Neurons per Session')

# 5. R² improvement by session
ax5 = axes[1, 1]
multi_valid.boxplot(column='delta_r2', by='session_num', ax=ax5)
ax5.set_xlabel('Session')
ax5.set_ylabel('ΔR² (Multi - Single)')
ax5.set_title('Fit Improvement Across Sessions')
ax5.axhline(0, color='r', linestyle='--', alpha=0.5)
plt.sca(ax5)
plt.xticks(rotation=0)

# 6. Firing rate by session
ax6 = axes[1, 2]
combined_results.boxplot(column='firing_rate', by='session_num', ax=ax6)
ax6.set_xlabel('Session')
ax6.set_ylabel('Firing Rate (Hz)')
ax6.set_title('Firing Rate Across Sessions')
ax6.set_yscale('log')
plt.sca(ax6)
plt.xticks(rotation=0)

plt.tight_layout()
plt.show()

print("\n✓ Analysis complete!")